In [32]:
import io
import zipfile
import requests
import frontmatter
import re

def read_repo_data(repo_owner,repo_name):
    """
    Download and parse all markdown files from github repository.
    Args:
        repo_owner: Github username and organization
        repo_name: Repository name
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download repository : {resp.status_code}")
    repository_data = []
    zf=zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not(filename_lower.endswith('md') 
            or(filename_lower.endswith('mdx'))):
            continue;

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8',errors='ignore')
                post=frontmatter.loads(content)
                data=post.to_dict()
                data['filename']=filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename} : {e}")
            continue
    zf.close()
    return repository_data
    
evidentlyai_docs = read_repo_data('evidentlyai','docs')
print(f"Evidently ai docs : {len(evidentlyai_docs)}")

def sliding_window(seq,size,step):
    """ Simple chunking """
    if size<=0 or step<=0:
        raise ValueError("size and step must be positive")
    if size<=step:
        raise ValueError("size should be more than step")
    n = len(seq)
    result = []
    for i in range(0,n,step):
        chunk=seq[i:i+size]
        result.append({'start':i,'chunk':chunk})
        if i+size>=n:
            break
    return result

# # Start - Simple chunking 
# evidently_chunks = []
# for doc in evidentlyai_docs:
   
#     doc_copy=doc.copy()
#     doc_content=doc_copy.pop('content')
#     chunks=sliding_window(doc_content,2000,1000)
#     for chunk in chunks:
#         chunk.update(doc_copy)
#     evidently_chunks.extend(chunks)
# print(len(evidently_chunks)) 
# #End - Simple chunking 



Evidently ai docs : 95
